# Imports

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchaudio
from torch import Tensor
from tqdm import tqdm
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Set Device

In [2]:
device1 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device:{device1}")

Using device:cuda:0


# Model Definition

In [3]:
import random
from typing import Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()

        # attention map
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_weight = self._init_new_params(out_dim, 1)

        # project
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)

        # batch norm
        self.bn = nn.BatchNorm1d(out_dim)

        # dropout for inputs
        self.input_drop = nn.Dropout(p=0.2)

        # activate
        self.act = nn.SELU(inplace=True)

        # temperature
        self.temp = 1.
        if "temperature" in kwargs:
            self.temp = kwargs["temperature"]

    def forward(self, x):
        
        # apply input dropout
        x = self.input_drop(x)

        # derive attention map
        att_map = self._derive_att_map(x)

        # projection
        x = self._project(x, att_map)

        # apply batch norm
        x = self._apply_BN(x)
        x = self.act(x)
        return x

    def _pairwise_mul_nodes(self, x):
      

        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        x_mirror = x.transpose(1, 2)

        return x * x_mirror

    def _derive_att_map(self, x):
       
        att_map = self._pairwise_mul_nodes(x)
        # size: (#bs, #node, #node, #dim_out)
        att_map = torch.tanh(self.att_proj(att_map))
        # size: (#bs, #node, #node, 1)
        att_map = torch.matmul(att_map, self.att_weight)

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)

        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        x = x.view(org_size)

        return x

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class HtrgGraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()

        self.proj_type1 = nn.Linear(in_dim, in_dim)
        self.proj_type2 = nn.Linear(in_dim, in_dim)

        # attention map
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_projM = nn.Linear(in_dim, out_dim)

        self.att_weight11 = self._init_new_params(out_dim, 1)
        self.att_weight22 = self._init_new_params(out_dim, 1)
        self.att_weight12 = self._init_new_params(out_dim, 1)
        self.att_weightM = self._init_new_params(out_dim, 1)

        # project
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)

        self.proj_with_attM = nn.Linear(in_dim, out_dim)
        self.proj_without_attM = nn.Linear(in_dim, out_dim)

        # batch norm
        self.bn = nn.BatchNorm1d(out_dim)

        # dropout for inputs
        self.input_drop = nn.Dropout(p=0.2)

        # activate
        self.act = nn.SELU(inplace=True)

        # temperature
        self.temp = 1.
        if "temperature" in kwargs:
            self.temp = kwargs["temperature"]

    def forward(self, x1, x2, master=None):
       
        num_type1 = x1.size(1)
        num_type2 = x2.size(1)

        x1 = self.proj_type1(x1)
        x2 = self.proj_type2(x2)

        x = torch.cat([x1, x2], dim=1)

        if master is None:
            master = torch.mean(x, dim=1, keepdim=True)

        # apply input dropout
        x = self.input_drop(x)

        # derive attention map
        att_map = self._derive_att_map(x, num_type1, num_type2)

        # directional edge for master node
        master = self._update_master(x, master)

        # projection
        x = self._project(x, att_map)

        # apply batch norm
        x = self._apply_BN(x)
        x = self.act(x)

        x1 = x.narrow(1, 0, num_type1)
        x2 = x.narrow(1, num_type1, num_type2)

        return x1, x2, master

    def _update_master(self, x, master):

        att_map = self._derive_att_map_master(x, master)
        master = self._project_master(x, master, att_map)

        return master

    def _pairwise_mul_nodes(self, x):
        

        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        x_mirror = x.transpose(1, 2)

        return x * x_mirror

    def _derive_att_map_master(self, x, master):
        
        att_map = x * master
        att_map = torch.tanh(self.att_projM(att_map))

        att_map = torch.matmul(att_map, self.att_weightM)

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _derive_att_map(self, x, num_type1, num_type2):
        
        att_map = self._pairwise_mul_nodes(x)
        
        att_map = torch.tanh(self.att_proj(att_map))
        

        att_board = torch.zeros_like(att_map[:, :, :, 0]).unsqueeze(-1)

        att_board[:, :num_type1, :num_type1, :] = torch.matmul(
            att_map[:, :num_type1, :num_type1, :], self.att_weight11)
        att_board[:, num_type1:, num_type1:, :] = torch.matmul(
            att_map[:, num_type1:, num_type1:, :], self.att_weight22)
        att_board[:, :num_type1, num_type1:, :] = torch.matmul(
            att_map[:, :num_type1, num_type1:, :], self.att_weight12)
        att_board[:, num_type1:, :num_type1, :] = torch.matmul(
            att_map[:, num_type1:, :num_type1, :], self.att_weight12)

        att_map = att_board

        # att_map = torch.matmul(att_map, self.att_weight12)

        # apply temperature
        att_map = att_map / self.temp

        att_map = F.softmax(att_map, dim=-2)

        return att_map

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)

        return x1 + x2

    def _project_master(self, x, master, att_map):

        x1 = self.proj_with_attM(torch.matmul(
            att_map.squeeze(-1).unsqueeze(1), x))
        x2 = self.proj_without_attM(master)

        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        x = x.view(org_size)

        return x

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class GraphPool(nn.Module):
    def __init__(self, k: float, in_dim: int, p: Union[float, int]):
        super().__init__()
        self.k = k
        self.sigmoid = nn.Sigmoid()
        self.proj = nn.Linear(in_dim, 1)
        self.drop = nn.Dropout(p=p) if p > 0 else nn.Identity()
        self.in_dim = in_dim

    def forward(self, h):
        Z = self.drop(h)
        weights = self.proj(Z)
        scores = self.sigmoid(weights)
        new_h = self.top_k_graph(scores, h, self.k)

        return new_h

    def top_k_graph(self, scores, h, k):
        
        _, n_nodes, n_feat = h.size()
        n_nodes = max(int(n_nodes * k), 1)
        _, idx = torch.topk(scores, n_nodes, dim=1)
        idx = idx.expand(-1, -1, n_feat)

        h = h * scores
        h = torch.gather(h, 1, idx)

        return h


class CONV(nn.Module):
    @staticmethod
    def to_mel(hz):
        return 2595 * np.log10(1 + hz / 700)

    @staticmethod
    def to_hz(mel):
        return 700 * (10**(mel / 2595) - 1)

    def __init__(self,
                 out_channels,
                 kernel_size,
                 sample_rate=16000,
                 in_channels=1,
                 stride=1,
                 padding=0,
                 dilation=1,
                 bias=False,
                 groups=1,
                 mask=False):
        super().__init__()
        if in_channels != 1:

            msg = "SincConv only support one input channel (here, in_channels = {%i})" % (
                in_channels)
            raise ValueError(msg)
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Forcing the filters to be odd ( perfectly symmetrics)
        if kernel_size % 2 == 0:
            self.kernel_size = self.kernel_size + 1
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.mask = mask
        if bias:
            raise ValueError('SincConv does not support bias.')
        if groups > 1:
            raise ValueError('SincConv does not support groups.')

        NFFT = 512
        f = int(self.sample_rate / 2) * np.linspace(0, 1, int(NFFT / 2) + 1)
        fmel = self.to_mel(f)
        fmelmax = np.max(fmel)
        fmelmin = np.min(fmel)
        filbandwidthsmel = np.linspace(fmelmin, fmelmax, self.out_channels + 1)
        filbandwidthsf = self.to_hz(filbandwidthsmel)

        self.mel = filbandwidthsf
        self.hsupp = torch.arange(-(self.kernel_size - 1) / 2,
                                  (self.kernel_size - 1) / 2 + 1)
        self.band_pass = torch.zeros(self.out_channels, self.kernel_size)
        for i in range(len(self.mel) - 1):
            fmin = self.mel[i]
            fmax = self.mel[i + 1]
            hHigh = (2*fmax/self.sample_rate) * \
                np.sinc(2*fmax*self.hsupp/self.sample_rate)
            hLow = (2*fmin/self.sample_rate) * \
                np.sinc(2*fmin*self.hsupp/self.sample_rate)
            hideal = hHigh - hLow

            self.band_pass[i, :] = Tensor(np.hamming(
                self.kernel_size)) * Tensor(hideal)

    def forward(self, x, mask=False):
        band_pass_filter = self.band_pass.clone().to(x.device)
        if mask:
            A = np.random.uniform(0, 20)
            A = int(A)
            A0 = random.randint(0, band_pass_filter.shape[0] - A)
            band_pass_filter[A0:A0 + A, :] = 0
        else:
            band_pass_filter = band_pass_filter

        self.filters = (band_pass_filter).view(self.out_channels, 1,
                                               self.kernel_size)

        return F.conv1d(x,
                        self.filters,
                        stride=self.stride,
                        padding=self.padding,
                        dilation=self.dilation,
                        bias=None,
                        groups=1)


class Residual_block(nn.Module):
    def __init__(self, nb_filts, first=False):
        super().__init__()
        self.first = first

        if not self.first:
            self.bn1 = nn.BatchNorm2d(num_features=nb_filts[0])
        self.conv1 = nn.Conv2d(in_channels=nb_filts[0],
                               out_channels=nb_filts[1],
                               kernel_size=(2, 3),
                               padding=(1, 1),
                               stride=1)
        self.selu = nn.SELU(inplace=True)

        self.bn2 = nn.BatchNorm2d(num_features=nb_filts[1])
        self.conv2 = nn.Conv2d(in_channels=nb_filts[1],
                               out_channels=nb_filts[1],
                               kernel_size=(2, 3),
                               padding=(0, 1),
                               stride=1)

        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv2d(in_channels=nb_filts[0],
                                             out_channels=nb_filts[1],
                                             padding=(0, 1),
                                             kernel_size=(1, 3),
                                             stride=1)

        else:
            self.downsample = False
        self.mp = nn.MaxPool2d((1, 3))  # self.mp = nn.MaxPool2d((1,4))

    def forward(self, x):
        identity = x
        if not self.first:
            out = self.bn1(x)
            out = self.selu(out)
        else:
            out = x
        out = self.conv1(x)

        # print('out',out.shape)
        out = self.bn2(out)
        out = self.selu(out)
        # print('out',out.shape)
        out = self.conv2(out)
        #print('conv2 out',out.shape)
        if self.downsample:
            identity = self.conv_downsample(identity)

        out += identity
        out = self.mp(out)
        return out


class AASISTModel(nn.Module):  # Changed from 'class Model' to avoid generic naming conflicts
    def __init__(self, d_args: dict, num_classes: int):
        super().__init__()

        self.d_args = d_args
        filts = d_args["filts"]
        gat_dims = d_args["gat_dims"]
        pool_ratios = d_args["pool_ratios"]
        temperatures = d_args["temperatures"]

        self.conv_time = CONV(out_channels=filts[0],
                              kernel_size=d_args["first_conv"],
                              in_channels=1)
        self.first_bn = nn.BatchNorm2d(num_features=1)

        self.drop = nn.Dropout(0.5, inplace=True)
        self.drop_way = nn.Dropout(0.2, inplace=True)
        self.selu = nn.SELU(inplace=True)

        self.encoder = nn.Sequential(
            nn.Sequential(Residual_block(nb_filts=filts[1], first=True)),
            nn.Sequential(Residual_block(nb_filts=filts[2])),
            nn.Sequential(Residual_block(nb_filts=filts[3])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])))

        self.pos_S = nn.Parameter(torch.randn(1, 23, filts[-1][-1]))
        self.master1 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))
        self.master2 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))

        self.GAT_layer_S = GraphAttentionLayer(filts[-1][-1],
                                               gat_dims[0],
                                               temperature=temperatures[0])
        self.GAT_layer_T = GraphAttentionLayer(filts[-1][-1],
                                               gat_dims[0],
                                               temperature=temperatures[1])

        self.HtrgGAT_layer_ST11 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])
        self.HtrgGAT_layer_ST12 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])

        self.HtrgGAT_layer_ST21 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])

        self.HtrgGAT_layer_ST22 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])

        self.pool_S = GraphPool(pool_ratios[0], gat_dims[0], 0.3)
        self.pool_T = GraphPool(pool_ratios[1], gat_dims[0], 0.3)
        self.pool_hS1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)

        self.pool_hS2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)

        self.out_layer = nn.Linear(5 * gat_dims[1], num_classes)

    def forward(self, x, Freq_aug=False):
        x = x.unsqueeze(1)
        x = self.conv_time(x, mask=Freq_aug)
        x = x.unsqueeze(dim=1)
        x = F.max_pool2d(torch.abs(x), (3, 3))
        x = self.first_bn(x)
        x = self.selu(x)

        # getting embeddings using encoder
        e = self.encoder(x)

        # spectral GAT (GAT-S)
        e_S, _ = torch.max(torch.abs(e), dim=3)  # max along time
        e_S = e_S.transpose(1, 2) + self.pos_S

        gat_S = self.GAT_layer_S(e_S)
        out_S = self.pool_S(gat_S)

        # temporal GAT (GAT-T)
        e_T, _ = torch.max(torch.abs(e), dim=2)  # max along freq
        e_T = e_T.transpose(1, 2)

        gat_T = self.GAT_layer_T(e_T)
        out_T = self.pool_T(gat_T)

        # learnable master node
        master1 = self.master1.expand(x.size(0), -1, -1)
        master2 = self.master2.expand(x.size(0), -1, -1)

        # inference 1
        out_T1, out_S1, master1 = self.HtrgGAT_layer_ST11(
            out_T, out_S, master=self.master1)

        out_S1 = self.pool_hS1(out_S1)
        out_T1 = self.pool_hT1(out_T1)

        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST12(
            out_T1, out_S1, master=master1)
        out_T1 = out_T1 + out_T_aug
        out_S1 = out_S1 + out_S_aug
        master1 = master1 + master_aug

        # inference 2
        out_T2, out_S2, master2 = self.HtrgGAT_layer_ST21(
            out_T, out_S, master=self.master2)
        out_S2 = self.pool_hS2(out_S2)
        out_T2 = self.pool_hT2(out_T2)

        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST22(
            out_T2, out_S2, master=master2)
        out_T2 = out_T2 + out_T_aug
        out_S2 = out_S2 + out_S_aug
        master2 = master2 + master_aug

        out_T1 = self.drop_way(out_T1)
        out_T2 = self.drop_way(out_T2)
        out_S1 = self.drop_way(out_S1)
        out_S2 = self.drop_way(out_S2)
        master1 = self.drop_way(master1)
        master2 = self.drop_way(master2)

        out_T = torch.max(out_T1, out_T2)
        out_S = torch.max(out_S1, out_S2)
        master = torch.max(master1, master2)

        T_max, _ = torch.max(torch.abs(out_T), dim=1)
        T_avg = torch.mean(out_T, dim=1)

        S_max, _ = torch.max(torch.abs(out_S), dim=1)
        S_avg = torch.mean(out_S, dim=1)

        last_hidden = torch.cat(
            [T_max, T_avg, S_max, S_avg, master.squeeze(1)], dim=1)

        last_hidden = self.drop(last_hidden)
        output = self.out_layer(last_hidden)

        return output


# Utilities (padding+split+TnT)

In [4]:
from __future__ import annotations
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchaudio
from torch import Tensor
from tqdm import tqdm
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def pad_random(x: np.ndarray, max_len: int = 64600):
    
    x_len = x.shape[0]
    
    # if duration is already long enough
    if x_len >= max_len:
        # random part
        stt = np.random.randint(x_len - max_len + 1) 
        return x[stt:stt + max_len]

    # if too short,keep repeating
    num_repeats = int(max_len / x_len) + 1
    padded_x = np.tile(x, (num_repeats))[:max_len]
    return padded_x


from typing import Any, Tuple

class DeepFakeDataset(Dataset):
    def __init__(self, audio_dir: str, label_file_path: str, label_transform = None):
        self.labels_df = pd.read_csv(label_file_path)
        self.audio_dir = audio_dir
        self.label_transform = label_transform
        
        # official repository recommendations
        self.target_sample_rate = 16000  
        self.target_length = 64600       
        

    def __len__(self) -> int:
        return len(self.labels_df)
    
    def __getitem__(self, index) -> Tuple[Tensor, Any]:
        # Get filename and label
        file_name = self.labels_df.iloc[index, 0]
        label = self.labels_df.iloc[index, 1] 

        # load audio
        file_path = os.path.join(self.audio_dir, file_name)
        waveform, sample_rate = torchaudio.load(file_path)

        #  apply AASIST pre-processing
        
        #  resample to 16kHz
        if sample_rate != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=self.target_sample_rate
            )
            waveform = resampler(waveform)
            

        #  convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0)
        
        #  squeeze to 1D tensor
        waveform = waveform.squeeze() # Shape: [time]

        #  convert to numpy for padding
        waveform_np = waveform.numpy()

        #  apply their exact padding/cropping logic
        waveform_padded_np = pad_random(waveform_np, self.target_length)
        
        #  convert back to a float tensor
        final_waveform_tensor = torch.from_numpy(waveform_padded_np).float()
        
        # End of pre-processing part

        if self.label_transform:
            label = self.label_transform(label)
        
        final_label = torch.tensor(label).long() 

        return final_waveform_tensor, final_label

def train_aasist(model: AASISTModel, train_loader: DataLoader, loss_function, optimizer):
    size_dataset = len(train_loader.dataset)
    print(f"The size of the Dataset loaded: {size_dataset}")

    model.train() #  enabling dropout
    
    total_loss = 0.0

    for batch_idx, (waveform_batch, label_batch) in enumerate(tqdm(train_loader, desc="Training AASIST")):
        waveform_batch, label_batch = waveform_batch.to(device), label_batch.to(device)
        
        optimizer.zero_grad()
        
        
        pred_logits = model(waveform_batch) 
        
        # loss
        loss = loss_function(pred_logits, label_batch)

        # backward pass
        # compute gradients
        loss.backward()
        
        #  update weights
        optimizer.step()
        
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Average Training Loss: {avg_loss:.4f}")


def test_aasist(model: AASISTModel, loss_function, val_loader: DataLoader) -> tuple:
    
    model.eval()  # eval mode
    val_size = len(val_loader.dataset)
    print(f"The size of the validation set: {val_size}")

    all_preds = []
    all_labels = []
    
    test_loss = 0.0
    
    with torch.no_grad(): 
        for wave, label in tqdm(val_loader, desc="Validating"):
            wave, label = wave.to(device), label.to(device)
            
            
            pred_logits = model(wave) 

            # loss
            loss = loss_function(pred_logits, label)
            test_loss += loss.item()

            
            # get the predicted class index
            predicted_class = pred_logits.argmax(dim=1)
            
            all_preds.append(predicted_class.cpu())
            all_labels.append(label.cpu())

    # average loss
    avg_loss = test_loss / len(val_loader)
    
    # concatenate all batches into single tensors
    y_pred = torch.cat(all_preds)
    y_true = torch.cat(all_labels)

    return avg_loss, y_pred, y_true

def get_report(y_true: torch.Tensor, y_pred: torch.Tensor):
    """
    Prints a full classification report and confusion matrix.
    
    Returns:
        - macro_f1 (float): The macro F1 score, for model checkpointing.
    """
    
    # class labels are 0, 1, 2, 3
    target_names = ['Class 0 (Real)', 'Class 1 (RealDist)', 'Class 2 (Fake)', 'Class 3 (FakeDist)']
    
    #  macro F1 
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # the full classification report 
    report = classification_report(y_true, y_pred, target_names=target_names, zero_division=0)
    
    # the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    print("\n--- Validation Report ---")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print("\nClassification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)
    print("-------------------------\n")
    
    return macro_f1


def run_epochs(model: AASISTModel, 
               train_loader: DataLoader,  
               loss_function, 
               optimizer, 
               scheduler,
               out_dir,
               num_epochs: int = 10,
               start_epoch: int = 0,
               model_prefix: str = "",
               val_loader: DataLoader = None):
   
    
    best_macro_f1 = 0.0  # tracking the best F1 score
    
    for epoch in range(start_epoch, start_epoch + num_epochs):
        print(f"--- Epoch {epoch+1}/{start_epoch + num_epochs} ---")
        
        #training
        train_aasist(model, train_loader, loss_function, optimizer)

        scheduler.step()

        if val_loader is not None:
            #  Val
            avg_val_loss, y_pred, y_true = test_aasist(model, loss_function, val_loader)
            
            print(f"Average Validation Loss: {avg_val_loss:.4f}")
            
            # reporting
            current_macro_f1 = get_report(y_true, y_pred)
            
            # checkpointing
            if current_macro_f1 > best_macro_f1:
                best_macro_f1 = current_macro_f1
                print(f" New best model! Macro F1: {best_macro_f1:.4f}")
                print("Saving model to 'best_model.pth'...")
                torch.save(model.state_dict(), os.path.join(out_dir,f"{model_prefix}_best_model.pth"))
                torch.save(model.state_dict(), os.path.join(out_dir,f"{model_prefix}_model_epoch{epoch+1}.pth"))
        else:
            torch.save(model.state_dict(), os.path.join(out_dir,f"{model_prefix}_model_epoch{epoch+1}.pth"))
        
    print("Training complete.")
    if val_loader is not None:
        print(f"Best Macro F1 achieved: {best_macro_f1:.4f}")

# Directory Setup

In [5]:
INPUT_ROOT = r"/kaggle/input/deep-fake-comsys-hackathon-6-task-a"
OUTPUT_ROOT = r"/kaggle/working"

In [6]:
if os.path.isdir(os.path.join(OUTPUT_ROOT,"saved_models")):
    print("Save Directory for Models Exists")
else:
    os.makedirs(os.path.join(OUTPUT_ROOT,"saved_models"))
    print("Save directory for models created successfully")

Save directory for models created successfully


In [7]:
print(os.listdir(INPUT_ROOT))
print(os.listdir(OUTPUT_ROOT))

['submission.csv', 'test_no_answer.csv', 'train.csv', 'test', 'train']
['saved_models', '.virtual_documents']


In [8]:
train_dir = os.path.join(INPUT_ROOT,"train")
test_dir = os.path.join(INPUT_ROOT,"test")
train_label_path = os.path.join(INPUT_ROOT,"train.csv")
model_save_dir = os.path.join(OUTPUT_ROOT,"saved_models")
BASE_MODEL_PATH = r"/kaggle/input/aasist/pytorch/large_multiclass/4/comsys_aasist_model_epoch9.pth"

# Setup Test and Validation Dataloaders

In [ ]:
audio_dataset = DeepFakeDataset(audio_dir=train_dir, label_file_path=train_label_path)

all_labels = audio_dataset.labels_df.iloc[:, 1].values

all_indices = list(range(len(audio_dataset)))

In [10]:
split_ratio = 0.9
val_split_ratio = 1.0 - split_ratio 


train_indices, val_indices = train_test_split(
    all_indices,
    test_size=val_split_ratio, 
    stratify=all_labels,      
    random_state=42          
)

print(f"Total samples:   {len(audio_dataset)}")
print(f"Training samples:  {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

Total samples:   47333
Training samples:  42599
Validation samples: 4734


In [11]:
train_dataset = Subset(audio_dataset, train_indices)
val_dataset = Subset(audio_dataset, val_indices)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,  # shuffling
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False, # shuffling
    num_workers=4
)

# Setup Model Configurations

In [13]:
model_config = {
    "first_conv": 128,
    "filts": [70, [1, 32], [32, 32], [32, 64], [64, 64]],
    "gat_dims": [64, 32],
    "pool_ratios": [0.5, 0.7, 0.5, 0.5],
    "temperatures": [2.0, 2.0, 100.0, 100.0]
}


new_model = AASISTModel(model_config, 4)

try:
    pretrained_weights = torch.load(BASE_MODEL_PATH)
    model_dict = new_model.state_dict()
    # filter out mismatching keys
    pretrained_dict = {k: v for k, v in pretrained_weights.items() if k in model_dict and "out_layer" not in k}
    model_dict.update(pretrained_dict)
    new_model.load_state_dict(model_dict)
    print(f"Model Weights Loaded Successfully from {BASE_MODEL_PATH}")
    
except Exception as e:
    print(f"Error loading weights: {e}")
    print("WARNING: Starting from scratch. This is NOT recommended.")

# compile
new_model = torch.compile(new_model)
new_model = new_model.to(device) 

print(new_model)

Error loading weights: [Errno 2] No such file or directory: '/kaggle/input/aasist/pytorch/large_multiclass/4/comsys_aasist_model_epoch9.pth'
OptimizedModule(
  (_orig_mod): AASISTModel(
    (conv_time): CONV()
    (first_bn): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (drop): Dropout(p=0.5, inplace=True)
    (drop_way): Dropout(p=0.2, inplace=True)
    (selu): SELU(inplace=True)
    (encoder): Sequential(
      (0): Sequential(
        (0): Residual_block(
          (conv1): Conv2d(1, 32, kernel_size=(2, 3), stride=(1, 1), padding=(1, 1))
          (selu): SELU(inplace=True)
          (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(32, 32, kernel_size=(2, 3), stride=(1, 1), padding=(0, 1))
          (conv_downsample): Conv2d(1, 32, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
          (mp): MaxPool2d(kernel_size=(1, 3), stride=(1, 3), padding=0, dilation=1, ceil_mode=False)
 

In [14]:
backbone_params = []
classifier_params = []
for name, param in new_model.named_parameters():
    if "out_layer" in name:
        classifier_params.append(param)
    else:
        backbone_params.append(param)

# Print total trainable parameters (nice quick check)
total_trainable = sum(p.numel() for p in new_model.parameters() if p.requires_grad)
print(f"Total trainable parameters in new_model: {total_trainable:,}")

Total trainable parameters in new_model: 298,188


# Training Logisitics

In [ ]:
LEARNING_RATE = 5 * 1e-5
EPOCHS = 15

# We can be a bit more confident with the LRs now
optimizer = torch.optim.Adam(new_model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

counts = torch.tensor([2512.0, 10049.0, 6954.0, 27818.0]).to(device1)
weights = 1.0 / counts
weights = weights / weights.sum()

loss_function = nn.CrossEntropyLoss(weight=weights)

## Logistics Setup1

In [18]:
run_epochs(model=new_model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    out_dir=model_save_dir,
    num_epochs=EPOCHS,
    scheduler=scheduler,
    model_prefix="h1"
)

--- Epoch 1/1 ---
The size of the Dataset loaded: 42599


Training AASIST:   0%|          | 0/1332 [00:00<?, ?it/s]W1201 16:01:24.516000 88 torch/_inductor/utils.py:1137] [0/1] Not enough SMs to use max_autotune_gemm mode
W1201 16:01:24.516000 88 torch/_inductor/utils.py:1137] [0/1] Not enough SMs to use max_autotune_gemm mode
Training AASIST: 100%|██████████| 1332/1332 [37:06<00:00,  1.67s/it]


Average Training Loss: 0.9238
The size of the validation set: 4734


Validating: 100%|██████████| 148/148 [01:40<00:00,  1.47it/s]

Average Validation Loss: 0.6334

--- Validation Report ---
Macro F1 Score: 0.6287

Classification Report:
                    precision    recall  f1-score   support

    Class 0 (Real)       0.45      0.82      0.58       251
Class 1 (RealDist)       0.39      1.00      0.56      1005
    Class 2 (Fake)       1.00      0.61      0.76       696
Class 3 (FakeDist)       1.00      0.45      0.62      2782

          accuracy                           0.61      4734
         macro avg       0.71      0.72      0.63      4734
      weighted avg       0.84      0.61      0.62      4734

Confusion Matrix:
[[ 205   45    1    0]
 [   0 1005    0    0]
 [ 247   19  424    6]
 [   0 1535    0 1247]]
-------------------------

 New best model! Macro F1: 0.6287
Saving model to 'best_model.pth'...
Training complete.
Best Macro F1 achieved: 0.6287


In [19]:
training_data = DeepFakeDataset(train_dir,train_label_path)
ent_loader = DataLoader(training_data)

In [ ]:
run_epochs(model=new_model,
    train_loader=ent_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    out_dir=model_save_dir,
    num_epochs=15,
    scheduler=scheduler,
    model_prefix="h2"
)

--- Epoch 1/1 ---
The size of the Dataset loaded: 47333


Training AASIST:  37%|███▋      | 17684/47333 [24:00<40:14, 12.28it/s]  



KeyboardInterrupt: 

In [21]:
avg_val_loss, y_pred, y_true = test_aasist(new_model, loss_function, val_loader)
            
print(f"Average Validation Loss: {avg_val_loss:.4f}")


current_macro_f1 = get_report(y_true, y_pred)

The size of the validation set: 4734


Validating: 100%|██████████| 148/148 [00:57<00:00,  2.57it/s]

Average Validation Loss: 1.0367

--- Validation Report ---
Macro F1 Score: 0.4083

Classification Report:
                    precision    recall  f1-score   support

    Class 0 (Real)       0.18      0.98      0.31       251
Class 1 (RealDist)       0.37      0.67      0.47      1005
    Class 2 (Fake)       0.48      0.79      0.60       696
Class 3 (FakeDist)       1.00      0.15      0.26      2782

          accuracy                           0.40      4734
         macro avg       0.51      0.65      0.41      4734
      weighted avg       0.75      0.40      0.35      4734

Confusion Matrix:
[[ 245    1    5    0]
 [ 333  670    2    0]
 [ 144    0  552    0]
 [ 630 1159  586  407]]
-------------------------



In [22]:
class TestDeepFakeDataset(Dataset):
    def __init__(self, test_dir, target_sr=16000, target_len=64600):
        self.test_dir = test_dir
        self.files = sorted(os.listdir(test_dir))    
        self.target_sr = target_sr
        self.target_len = target_len

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        filepath = os.path.join(self.test_dir, filename)

        # Load audio
        waveform, sr = torchaudio.load(filepath)

        # 1. Resample if necessary (Crucial for AASIST)
        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            waveform = resampler(waveform)
        
        # 2. Convert to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0)
            
        # 3. Squeeze
        waveform = waveform.squeeze()

        # 4. Pad/Crop (Using the pad_random function you defined in Utilities)
        waveform_np = waveform.numpy()
        waveform_padded_np = pad_random(waveform_np, self.target_len)
        final_waveform_tensor = torch.from_numpy(waveform_padded_np).float()

        return final_waveform_tensor, filename

In [26]:
def generate_submission(config, model_weights_path, test_dir, output_csv_path, device):
    print(f"--- Starting Inference ---")
    
    # 1. Initialize Dataset & Loader
    # Using target_len=64600 as per your previous dataset class defaults
    test_ds = TestDeepFakeDataset(test_dir, target_sr=16000, target_len=64600)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=4)
    
    # 2. Initialize Model Architecture
    model = AASISTModel(config, num_classes=4)
    model.to(device)
    
    # 3. Load Weights (Robust Loading Fix)
    if os.path.exists(model_weights_path):
        print(f"Loading weights from: {model_weights_path}")
        try:
            state_dict = torch.load(model_weights_path, map_location=device)
            
            # --- FIX STARTS HERE ---
            new_state_dict = {}
            for k, v in state_dict.items():
                # Remove '_orig_mod.' prefix added by torch.compile
                name = k.replace("_orig_mod.", "")
                # Remove 'module.' prefix added by DataParallel (just in case)
                name = name.replace("module.", "")
                new_state_dict[name] = v
            # --- FIX ENDS HERE ---
            
            model.load_state_dict(new_state_dict)
            print("Weights loaded successfully.")
            
        except Exception as e:
            print(f"Error loading weights: {e}")
            return
    else:
        print(f"Error: Model path {model_weights_path} does not exist.")
        return

    model.eval()
    
    all_filenames = []
    all_predictions = []
    
    # 4. Inference Loop
    print("Running prediction loop...")
    with torch.no_grad():
        for wave_batch, filename_batch in tqdm(test_loader, desc="Inference"):
            wave_batch = wave_batch.to(device)
            
            # Forward pass
            pred_logits = model(wave_batch)
            
            # Get class with highest probability
            predicted_classes = pred_logits.argmax(dim=1).cpu().numpy()
            
            all_predictions.extend(predicted_classes)
            all_filenames.extend(list(filename_batch))

    # 5. Save to CSV
    submission_df = pd.DataFrame({
        "File_name": all_filenames,
        "target": all_predictions
    })
    
    submission_df.to_csv(output_csv_path, index=False)
    print(f"✅ Submission saved to: {output_csv_path}")
    print(submission_df.head())

In [27]:
# --- Configuration ---
TEST_DIR = os.path.join(INPUT_ROOT, "test")
OUTPUT_CSV = os.path.join(OUTPUT_ROOT, "submission.csv")

BEST_MODEL_PATH = os.path.join(OUTPUT_ROOT, "saved_models", "h1_best_model.pth") 

# Check if we have a model to load, otherwise use the base one for testing
if not os.path.exists(BEST_MODEL_PATH):
    print(f"Warning: {BEST_MODEL_PATH} not found. Please update the path.")
    files = [f for f in os.listdir(os.path.join(OUTPUT_ROOT, "saved_models")) if f.endswith('.pth')]
    if files:
        BEST_MODEL_PATH = os.path.join(OUTPUT_ROOT, "saved_models", files[-1])
        print(f"Found alternative model: {BEST_MODEL_PATH}")

# --- Run ---
generate_submission(
    config=model_config,
    model_weights_path=BEST_MODEL_PATH,
    test_dir=TEST_DIR,
    output_csv_path=OUTPUT_CSV,
    device=device1
)

--- Starting Inference ---
Loading weights from: /kaggle/working/saved_models/h1_best_model.pth
Weights loaded successfully.
Running prediction loop...


Inference: 100%|██████████| 1268/1268 [05:48<00:00,  3.64it/s]

✅ Submission saved to: /kaggle/working/submission.csv
   File_name  target
0      1.wav       2
1     10.wav       2
2    100.wav       2
3   1000.wav       2
4  10000.wav       3
